# Joint Training: FRCRN + XLSR + AD Classifier

Pipeline: Raw Pitt Audio -> FRCRN Denoise -> XLSR-53 Feature Extraction -> AD Classification

Loss: L_total = alpha * L1_denoise + beta * CE_classify

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np

# Add joint_train and train to path
JOINT_TRAIN_DIR = Path.cwd() if Path.cwd().name == "joint_train" else Path(__file__).parent
sys.path.insert(0, str(JOINT_TRAIN_DIR))
sys.path.insert(0, str(JOINT_TRAIN_DIR.parent / "train"))  # for visualization, data_split

from joint_config import (
    PROJECT_ROOT, RANDOM_SEEDS, RANDOM_SEED, TRAIN_SET_RATIO,
    SAMPLING_RATE, JOINT_BATCH_SIZE, NUM_WORKERS,
)
from joint_dataset import create_joint_dataloaders
from joint_train import train
from data_split import create_split
from visualization import plot_training_curves

## Configuration

In [ ]:
# Paths
RAW_AUDIO_DIR = PROJECT_ROOT / "data/raw/Pitt"
CLEAN_AUDIO_DIR = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"  # pseudo clean target
MODEL_OUTPUT_DIR = PROJECT_ROOT / "models/joint_seed_42"

# FRCRN pretrained weights (download from ModelScope if needed)
FRCRN_PRETRAINED_PATH = PROJECT_ROOT / "models/speech_frcrn_ans_cirm_16k/pytorch_model.bin"

print(f"Raw audio:   {RAW_AUDIO_DIR}")
print(f"Clean audio: {CLEAN_AUDIO_DIR}")
print(f"Output:      {MODEL_OUTPUT_DIR}")
print(f"FRCRN ckpt:  {FRCRN_PRETRAINED_PATH}")

In [ ]:
# Device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using {device}")

## Step 1: Train/Validation Split

In [ ]:
# Reuse existing Pitt train/val split (same partition as frozen-XLSR experiments)
TRAIN_CSV, VAL_CSV = create_split("Pitt")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Val CSV:   {VAL_CSV}")

## Step 2: Create Data Loaders

In [ ]:
train_loader, val_loader = create_joint_dataloaders(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    raw_audio_dir=RAW_AUDIO_DIR,
    clean_audio_dir=CLEAN_AUDIO_DIR,
)

In [ ]:
# Class weights (handle imbalanced dataset)
num_control = sum(1 for _, l in train_loader.dataset.samples if l == 0)
num_dementia = sum(1 for _, l in train_loader.dataset.samples if l == 1)
total = num_control + num_dementia

class_weight_control = total / (2 * num_control)
class_weight_dementia = total / (2 * num_dementia)

print(f"Control: {num_control}, Dementia: {num_dementia}")
print(f"Class weights: Control={class_weight_control:.4f}, Dementia={class_weight_dementia:.4f}")

## Step 3: Joint Training (Seed 42)

In [ ]:
frcrn_path = str(FRCRN_PRETRAINED_PATH) if FRCRN_PRETRAINED_PATH.exists() else None
if frcrn_path is None:
    print("WARNING: FRCRN pretrained weights not found, training from scratch")

seed, best_metrics, history = train(
    seed=42,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device,
    frcrn_pretrained_path=frcrn_path,
    class_weight_control=class_weight_control,
    class_weight_dementia=class_weight_dementia,
)

In [ ]:
plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix='Joint Training - Seed 42'
)

## Step 4: Results

In [ ]:
print(f"Best Validation Results (Seed {seed}):")
print(f"  Accuracy:     {best_metrics['accuracy'] * 100:.2f}%")
print(f"  F1 Score:     {best_metrics['f1_score']:.4f}")
print(f"  Control Acc:  {best_metrics['control_acc'] * 100:.2f}%")
print(f"  Dementia Acc: {best_metrics['dementia_acc'] * 100:.2f}%")
print(f"  Val Loss:     {best_metrics['loss']:.4f}")
print(f"  Denoise Loss: {best_metrics['denoise_loss']:.4f}")
print(f"  Classify Loss:{best_metrics['classify_loss']:.4f}")

print(f"\nModels saved to: {MODEL_OUTPUT_DIR}")
print(f"  FRCRN:    frcrn_best.pth")
print(f"  XLSR:     xlsr_best.pth")
print(f"  AD Model: ad_model_best.pth")
print(f"  Meta:     meta.pth")

In [ ]:
torch.cuda.empty_cache()